# Установка моделей

In [1]:
from pprint import pprint

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer

/home/misha/Desktop/knowledge_distillation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Установка токенизаторов

In [2]:
model_name_1 = "LiquidAI/LFM2.5-230M"
model_name_2 = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer_1 = AutoTokenizer.from_pretrained(model_name_1)
model_1 = AutoModelForCausalLM.from_pretrained(model_name_1,
                                               torch_dtype=torch.float16,
                                               device_map="auto")

tokenizer_2 = AutoTokenizer.from_pretrained(model_name_2)
model_2 = AutoModelForCausalLM.from_pretrained(model_name_2,
                                               torch_dtype=torch.float16,
                                               device_map="auto")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 2874.26it/s]


# Класс генерации forward pass'а

In [3]:
'' == None

False

In [4]:
from transformers.generation import GenerationMixin
from transformers.tokenization_utils_sentencepiece import SentencePieceBackend
from transformers.tokenization_utils_tokenizers import TokenizersBackend

TokenizerType = TokenizersBackend | SentencePieceBackend

class Generation:
    def __init__(
        self,
        # llm_1: GenerationMixin,
        # llm_2: GenerationMixin,
        llms: list[GenerationMixin],
        tokenizers: list[TokenizerType],
        # tok_2: TokenizerType,
        top_k: int,
    ):
        self.llms = llms
        self.tokenizers = tokenizers
        self.chat_prefixes = []
        # self.llm_1 = llm_1
        # self.llm_2 = llm_2
        # self.tok_1 = tok_1
        # self.tok_2 = tok_2
        self.top_k = top_k
        self.device_1 = next(llms[0].parameters()).device
        self.device_2 = next(llms[0].parameters()).device
        # self.chat_prefix_1: str | None = None
        # self.chat_prefix_2: str | None = None

    @staticmethod
    def _build_chat_prefix(
        tokenizer: TokenizerType,
        user_message: str,
    ) -> str:
        messages = [
            {
                "role": "user",
                "content": user_message,
            }
        ]

        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    @staticmethod
    def _tokenize_prompt(
        tokenizer: TokenizerType,
        prompt: str,
        device: torch.device,
    ):
        return tokenizer(
            prompt,
            return_tensors="pt",
            add_special_tokens=False,
        ).to(device)

    @staticmethod
    def _generate_log_probs(
        model: GenerationMixin,
        inputs,
    ) -> torch.Tensor:
        with torch.inference_mode():
            logits = model(**inputs).logits[:, -1, :]

        return torch.log_softmax(
            logits.float(),
            dim=-1,
        )

    def _get_top_distribution(
        self,
        log_probs: torch.Tensor,
        tokenizer: TokenizerType,
        model_number: int,
    ) -> dict[str, float]:
        k = min(self.top_k, log_probs.shape[-1])

        top_log_probs, top_token_ids = torch.topk(
            log_probs[0],
            k=k,
        )

        distribution: dict[str, float] = {}

        print(f"Модель {model_number}")

        for rank, (token_id, log_prob) in enumerate(
            zip(top_token_ids, top_log_probs, strict=True),
            start=1,
        ):
            token_id_int = token_id.item()

            token_text = tokenizer.decode(
                [token_id_int],
                skip_special_tokens=False,
            )

            probability = log_prob.exp().item()

            print(
                f"{rank}. "
                f"token={token_text!r}, "
                f"id={token_id_int}, "
                f"prob={probability:.6f}"
            )

            distribution[token_text] = (
                distribution.get(token_text, 0.0)
                + probability
            )

        return distribution

    def initialize_chat(self, user_message: str) -> None:
        for tokenizer in self.tokenizers:
            chat_prefix = self._build_chat_prefix(tokenizer=tokenizer,
                                                  user_message=user_message,)
            self.chat_prefixes.append(chat_prefix)

            # self.chat_prefix_2 = self._build_chat_prefix(
            #     tokenizer=self.tok_2,
            #     user_message=user_message,
            # )

            # print("Конец chat-prefix модели 1:")
            # print(repr(self.chat_prefix_1[-150:]))

            # print("Конец chat-prefix модели 2:")
            # print(repr(self.chat_prefix_2[-150:]))

    def generate_pipe(
        self,
        generated_text: str,
        model_to_run: str = '',
    ):
        if self.chat_prefixes is None:
            raise RuntimeError(
                "Сначала вызови initialize_chat(user_message)"
            )


        print(f'Внутри generate pipe {model_to_run=}')
        distributions: dict[str, dict[str, float]] = {}
        if model_to_run == "":
            print('попали в условие запуска всех моделей')
            for idx, (llm, tokenizer) in enumerate(zip(self.llms, self.tokenizers, strict=True)):
                prompt = self.chat_prefixes[idx] + generated_text
                inputs = self._tokenize_prompt(
                    tokenizer=tokenizer,
                    prompt=prompt,
                    device=self.device_1,
                )

                log_probs = self._generate_log_probs(model=llm,
                                                     inputs=inputs)

                distributions[idx] = self._get_top_distribution(log_probs=log_probs,
                                                                tokenizer=tokenizer,
                                                                model_number=idx,)

                # prompt_2 = self.chat_prefix_2 + generated_text
                # inputs_2 = self._tokenize_prompt(
                #     tokenizer=self.tok_2,
                #     prompt=prompt_2,
                #     device=self.device_2,
                # )

                # log_probs_2 = self._generate_log_probs(
                #     model=self.llm_2,
                #     inputs=inputs_2,
                # )

                # distributions["2"] = self._get_top_distribution(
                #     log_probs=log_probs_2,
                #     tokenizer=self.tok_2,
                #     model_number=2,
                # )

        if model_to_run != "":
            print(f'{model_to_run=}')
            prompt = self.chat_prefixes[int(model_to_run)] + generated_text
            print(f'попали в условие запуска модели {model_to_run}')
            inputs = self._tokenize_prompt(tokenizer=self.tokenizers[model_to_run],
                                           prompt=prompt,
                                           device=self.device_1)

            log_probs = self._generate_log_probs(model=self.llms[model_to_run],
                                                 inputs=inputs,)

            distributions[model_to_run] = self._get_top_distribution(log_probs=log_probs,
                                                                     tokenizer=self.tokenizers[model_to_run],
                                                                     model_number=model_to_run,)

        # if model_to_run == "model_1":
        #     prompt_2 = self.chat_prefix_2 + generated_text
        #     print('попали в условие запуска модели 1')
        #     inputs_2 = self._tokenize_prompt(tokenizer=self.tok_2,
        #                                      prompt=prompt_2,
        #                                      device=self.device_2,)

        #     log_probs_2 = self._generate_log_probs(model=self.llm_2,
        #                                            inputs=inputs_2,)

        #     distributions["2"] = self._get_top_distribution(log_probs=log_probs_2,
        #                                                     tokenizer=self.tok_2,
        #                                                     model_number=2,)

        # if len(distributions) != 1:

        # if "1" in distributions and "2" in distributions:
        #     return distributions["1"], distributions["2"]

        # if "1" in distributions:
        #     return distributions["1"]

        # if "2" in distributions:
        #     return distributions["2"]

        return distributions

# Класс префиксной плотности

In [ ]:
class PrefixDense:
    def __init__(self,
                 probs_generator: Generation,
                 input_str: str,
                 stop_token: str):
        self.prob_distribution_1 = {}
        self.prob_distribution_2 = {}
        self.prob_distributions = {}
        self.step_matrix = {}
        self.model_to_run = ''
        self.max_steps: int = 10
        self.probs_generator = probs_generator
        self.user_prompt = input_str
        self.generated_text = ""
        self.max_steps = 512
        self.stop_token = stop_token

    
    def runpipe(self):
        self.probs_generator.initialize_chat(
            user_message=self.user_prompt,
        )
        print(f'{self.max_steps}')
        for step in range(self.max_steps):
            print("###############################################################")
            print(f"Шаг: {step}")
            print("Пользовательский запрос:")
            print(self.user_prompt)
            print("Сгенерированное продолжение:")
            print(repr(self.generated_text))
            print("###############################################################")
            self.prob_distributions = self.probs_generator.generate_pipe(generated_text=self.generated_text)
            # self.prob_distribution_1, self.prob_distribution_2 = result
            print(f'{self.prob_distributions=}')
            # print(f'{self.prob_distribution_2=}')
            ### Блок сопоставления токенов ###
            selected_text = self.match_chars()
            self.generated_text += selected_text
            print(f"Выбранный фрагмент: {selected_text!r}")
            if self.stop_token in selected_text:
                break
            print(f"Текущий ответ: {self.generated_text!r}")
        return self.generated_text


    def count_min_token_len_per_distrib(self, distribs: dict):
        distrib_min_len = {}
        distrib_max_len = {}
        for model_num, distrib in distribs.items():
            distrib_min_len[model_num] = min({len(key) for key in distrib})
            distrib_max_len[model_num] = max({len(key) for key in distrib})
        return distrib_min_len, distrib_max_len


    def find_the_suitest_token(self, 
                               distrib: dict, 
                               char_num: int, 
                               distrib_num: int):
        print(f'Надо потестить {distrib_num=}')
        char_prob = {}
        new_distrib = {}
        print(f'Для теста {distrib=}')
        for token, probability in distrib.items():
            print(f"{char_num=}")
            print(f"{token=}")
            if char_num >= len(token):
                print(
                    f'Индекс {char_num} выходит за границы '
                    f'токена "{token}".'
                )
                result = self.probs_generator.generate_pipe(
                    generated_text=self.generated_text + token,
                    model_to_run=f"{distrib_num}",
                )
                if isinstance(result, (tuple, list)):
                    result = result[distrib_num]
                pprint(
                    f'Для токена "{token}" с вероятностью '
                    f"{probability} получили {result=}"
                )
                for token2, prob2 in result.items():
                    if not token2 or prob2 <= 0:
                        continue
                    ongoing_token = token + token2
                    ongoing_prob = probability * prob2
                    print(
                        f'Полученный токен "{ongoing_token}", '
                        f"вероятность={ongoing_prob}"
                    )
                    new_distrib[ongoing_token] = (
                        new_distrib.get(ongoing_token, 0.0)
                        + ongoing_prob
                    )
                    if char_num < len(ongoing_token):
                        current_char = ongoing_token[char_num]

                        char_prob[current_char] = (
                            char_prob.get(current_char, 0.0)
                            + ongoing_prob
                        )
            else:
                new_distrib[token] = (
                    new_distrib.get(token, 0.0)
                    + probability
                )
                current_char = token[char_num]
                char_prob[current_char] = (
                    char_prob.get(current_char, 0.0)
                    + probability
                )
            print(f'Для проверки содержиомго {char_prob=}')
            tmp_dict = {}
            for token, prob in char_prob.items():
                print(f'Вероятность токена "{token}" — {prob}')
                print(f'Сумма всех токенов по всему вероятностному распределению — {sum(distrib.values())}')
                tmp_dict[token] = prob/sum(distrib.values())
            print(f"После пересчета {tmp_dict}")

        if not char_prob:
            pass
        print(f"Первоначально {char_prob}")
        print(f"{new_distrib=}")
        the_most_common_char = max(
            tmp_dict,
            key=tmp_dict.get,
        ) if tmp_dict else ''
        return the_most_common_char, new_distrib
    
    def ensemble(self, ensemble_distr: dict):
        token_prob = {}
        for distrib in ensemble_distr.values():
            for token, prob in distrib.items():
                if token not in token_prob:
                    token_prob[token] = prob
                else:
                    token_prob[token] += prob
        
        the_most_popular_token = max(token_prob, key=token_prob.get)
        return the_most_popular_token

    def ensemble_str(self, ensemble_dict: dict, probs_list: list):
        ensembling_probs = {}
        print(f'здесь {ensemble_dict=}')
        for distr in ensemble_dict.values():
            print(f'{distr=}')
            for distribution in distr.values():
                if not isinstance(distribution, str):
                    for token, prob in distribution.items():
                        if token not in ensembling_probs:
                            ensembling_probs[token] = prob
                        else:
                            ensembling_probs[token] += prob
                else:
                    result = "".join(distr[key] for key in sorted(distr))
                    print(f'{result=}')
                    print(f'{probs_list=}')
                    for model_num, distr in probs_list.items():
                        print(f'{model_num=}')
                        for token, prob in distr.items():
                            if token.startswith(result):
                                print(f'Токен {token} начинается с {result}, добавляем его вероятность')
                                if result not in ensembling_probs:
                                    ensembling_probs[result] = prob
                                else:
                                    ensembling_probs[result] += prob        
        pprint(f'{ensembling_probs=}')
        most_likely_token = max(ensembling_probs, key=ensembling_probs.get)
        return most_likely_token
   

    def match_chars(self):
        probs_list = self.prob_distributions
        distrib_min_len, distrib_max_len = self.count_min_token_len_per_distrib(distribs=probs_list)
        # distrib_max_len = self.count_max_token_len_per_distrib(distribs=probs_list)

        ensemble_dict = {}
        print(f'{probs_list=}')
        print(f'{distrib_min_len=}')
        print(f'{distrib_max_len=}')
        prefix = ''
        max_char_steps = 128
        char_num = 0
        while char_num < max_char_steps:
            print(f'Внимание! {char_num=}')
            print(f'Итерируемся по такому распределению {probs_list=}')
            for_suitest = {}
            for distrib_num, distrib in probs_list.items():
                print(f'До раскрытия: {probs_list}')
                the_most_common_char, new_distrib = self.find_the_suitest_token(distrib=distrib,
                                                                                char_num=char_num,
                                                                                distrib_num=distrib_num) 
                print(f'После раскрытия  {new_distrib=}')
                for_suitest[distrib_num] = new_distrib
                print(f'{the_most_common_char=}')
                print(f'Ансамбль дикт на старте {ensemble_dict=}')
                if isinstance(the_most_common_char, dict): # Кейс, когда нет общих совпадений внутри распределений
                    print('перезаписываем1')
                    ensemble_dict[f'distrib_num_{distrib_num}'] = {char_num: the_most_common_char}
                elif isinstance(the_most_common_char, str): # Кейс, когда есть общие совпадения внутри распределений
                    if f'distrib_num_{distrib_num}' not in ensemble_dict:
                        print('перезаписываем')
                        ensemble_dict[f'distrib_num_{distrib_num}'] = {char_num: the_most_common_char}
                    else:
                        ensemble_dict[f'distrib_num_{distrib_num}'][char_num] = the_most_common_char

            print(f'Ансамбль дикт на финише {ensemble_dict=}')
            print(f'Финалисты {char_num}-того символа {ensemble_dict=}')
            probs_list = for_suitest
            most_likely_token = self.ensemble_str(ensemble_dict=ensemble_dict, probs_list=probs_list)
            prefix = prefix + most_likely_token[-1]
            print(f'{prefix=}')
            print(f'{most_likely_token=}')
            print(f'{probs_list=}')
            new_probs_dict = {}
            
            # Сортировка по релеватным для продолжения токенам
            print(f'Начинаем зачистку токенов, не начинающихся с "{prefix}"')
            pprint(f'Как было до: {probs_list=}')
            for model_num, elem in probs_list.items():
                tmp_distrib = {}
                for token, prob in elem.items():
                    if token.startswith(most_likely_token):
                        print(f'Токен "{token}" начинается с "{most_likely_token}", оставляем его в пуле на следующую проверку')
                        tmp_distrib[token] = prob
                new_probs_dict[model_num] = tmp_distrib
            
            print(f'Как стало после: {new_probs_dict=}')
            if all(current.keys() == list(new_probs_dict.values())[0].keys() for current in list(new_probs_dict.values())[1:]):
                summed_probs = {
                    key: sum(distrib[key] for distrib in new_probs_dict.values())
                    for key in list(new_probs_dict.values())[0]
                }

                return_string = max(summed_probs, key=summed_probs.get)

                print(f'Суммарные вероятности: {summed_probs}')
                print(f'Возвращаем {return_string}')

                return return_string

            for distrib in new_probs_dict.values():
                print(len(distrib))
                if len(distrib) == 1:
                    for string in distrib:
                        return_string = string
                    print('Длина полученного распределения — 1')
                    return return_string

            char_num += 1
            probs_list = new_probs_dict  

            new_ensemble_dict = {}
            for distrib_num, char_winner_dict in ensemble_dict.items():
                print(f'Полученное распределение {char_winner_dict=}')
                for char_num, char_winner in char_winner_dict.items():
                    tmp_dict = {}
                    tmp_dict[char_num] = most_likely_token
                    print(f'{tmp_dict=}')
                new_ensemble_dict[distrib_num] = tmp_dict

            ensemble_dict = new_ensemble_dict
            print(f'Сейчас {ensemble_dict=}')

                    

# Инициализация пробной задачи

In [18]:
task = 'A family of 12 monkeys collected 10 piles of bananas. 6 piles had 9 hands, with each hand having 14 bananas, while the remaining piles had 12 hands, with each hand having 9 bananas. How many bananas would each monkey get if they divide the bananas equally amongst themselves?'

# Тестирование Prefix-dense подхода

In [19]:
agg = Generation(
    # llm_1=model_1,
    # llm_2=model_2,
    llms=[model_1,model_2],
    tokenizers=[tokenizer_1,tokenizer_2],
    # tok_1=tokenizer_1,
    # tok_2=tokenizer_2,
    top_k=5,
)

agreement = PrefixDense(
    probs_generator=agg,
    input_str=task,
    stop_token='<|im_end|>'
)

answer = agreement.runpipe()

print("Итог:")
print(repr(answer))

512
###############################################################
Шаг: 0
Пользовательский запрос:
A family of 12 monkeys collected 10 piles of bananas. 6 piles had 9 hands, with each hand having 14 bananas, while the remaining piles had 12 hands, with each hand having 9 bananas. How many bananas would each monkey get if they divide the bananas equally amongst themselves?
Сгенерированное продолжение:
''
###############################################################
Внутри generate pipe model_to_run=''
попали в условие запуска всех моделей
Модель 0
1. token='First', id=9578, prob=0.733749
2. token='There', id=3776, prob=0.137867
3. token='To', id=3097, prob=0.049932
4. token='The', id=1098, prob=0.044759
5. token='Let', id=8232, prob=0.007900
Модель 1
1. token='To', id=1249, prob=0.883882
2. token='First', id=5338, prob=0.042652
3. token='Let', id=10061, prob=0.041340
4. token='Sure', id=39814, prob=0.007768
5. token='There', id=3862, prob=0.003556
self.prob_distributions={0: {'First'

TypeError: can only concatenate str (not "NoneType") to str

# Тестирование модели 1

In [ ]:


# Qwen/Qwen2.5-0.5B-Instruct

streamer = TextStreamer(tokenizer_1, skip_prompt=True, skip_special_tokens=True)
prompt = task
input_ids = tokenizer_1.apply_chat_template(
    [{"role": "user", "content": prompt}],
    add_generation_prompt=True,
    return_tensors="pt",
    tokenize=True,
)["input_ids"].to(model_1.device)

output = model_1.generate(
    input_ids,
    do_sample=True,
    temperature=0.1,
    top_k=50,
    repetition_penalty=1.05,
    max_new_tokens=512,
    streamer=streamer,
)

# Тестирование модели 2

In [ ]:
streamer = TextStreamer(tokenizer_2, skip_prompt=True, skip_special_tokens=True)

prompt = task

input_ids = tokenizer_2.apply_chat_template(
    [{"role": "user", "content": prompt}],
    add_generation_prompt=True,
    return_tensors="pt",
    tokenize=True,
)["input_ids"].to(model_2.device)

output = model_2.generate(
    input_ids,
    do_sample=True,
    temperature=0.1,
    top_k=50,
    repetition_penalty=1.05,
    max_new_tokens=512,
    streamer=streamer,
)

# Проверка на наборе данных

In [ ]:
import polars as pl

splits = {'train': 'main/train-00000-of-00001.parquet', 'test': 'main/test-00000-of-00001.parquet'}
df = pl.read_parquet("hf://datasets/openai/gsm8k/" + splits["train"])

In [ ]:
questions = df['question'].to_list()
answers = df['answer'].to_list()

In [ ]:
from enum import Enum

class WhichModel(Enum):
    model_1: int = 1
    model_2: int = 2

class Solver:
    def __init__(self,
                 streamer: TextStreamer, 
                 model_1: GenerationMixin,
                 model_2: GenerationMixin,
                 tokenizer_1: TokenizerType, 
                 tokenizer_2: TokenizerType,
                 topk: int):
        self.streamer = streamer
        self.model_1 = model_1
        self.model_2 = model_2
        self.tokenizer_1 = tokenizer_1
        self.tokenizer_2 = tokenizer_2
        self.topk = topk

    def solve_solo(self, which_model_generate: WhichModel, task: str):
        if which_model_generate == WhichModel.model_1.value:
            model = self.model_1
            tokenizer = self.tokenizer_1
        elif which_model_generate == WhichModel.model_2.value:
            model = self.model_2
            tokenizer = self.tokenizer_2
        input_ids = tokenizer.apply_chat_template([{"role": "user", "content": prompt}],
                                                    add_generation_prompt=True,
                                                    return_tensors="pt",
                                                    tokenize=True)["input_ids"].to(model.device)
        output = model.generate(
            input_ids,
            do_sample=True,
            temperature=0.1,
            top_k=50,
            repetition_penalty=1.05,
            max_new_tokens=512,
            streamer=streamer,
        )
        return output

    def solve_ensemble(self, task: str):
        agg = Generation(llm_1=self.model_1,
                         llm_2=self.model_2,
                         tok_1=self.tokenizer_1,
                         tok_2=self.tokenizer_2,
                         top_k=self.topk,)

        agreement = PrefixDense(
            probs_generator=agg,
            input_str=task,
            stop_token='<|im_end|>'
        )

        answer = agreement.runpipe()
        print("Итог:")
        print(repr(answer))
        return repr(answer)

In [ ]:
streamer = TextStreamer(tokenizer_2, skip_prompt=True, skip_special_tokens=True)
solver = Solver(streamer=streamer,
                model_1=model_1,
                model_2=model_2,
                tokenizer_1=tokenizer_1,
                tokenizer_2=tokenizer_2,
                topk=5)
delimiter = '#########################'
limit = 100

for question, answer in zip(questions[:limit], answers[:limit], strict=True):
    try:
        ensemble_result = solver.solve_ensemble(task=question)
        with open("ensemble_output.txt", "a", encoding="utf-8") as file:
            file.write(f"Right answer:\n{answer}\n")
            file.write(f"LLM's answer: {ensemble_result}\n")
            file.write(f"{delimiter}\n")
    except Exception as e:
        print(f'Возникла ошибка: {e}')
        continue